# Liveness 03 · Server Model (DINOv2 ViT-S + LoRA)

The server re-checks one frame for every check-in (spec section 10.2), so it can afford a bigger model.
**ViT-S/14 pre-trained with DINOv2** generalises best to unseen attack media and phones; fine-tune only
**LoRA adapters + the head** (≈1 % of the weights) to avoid over-fitting the small spoof datasets.
Input: face crop ×2.7, 224×224. Inference on GPU ≈ 5 ms.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Liveness
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import timm, torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=2, img_size=224)
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["qkv", "proj"], modules_to_save=["head"])
model = get_peft_model(backbone, lora).to(device)
model.print_trainable_parameters()

In [ ]:
cfg = timm.data.resolve_data_config({}, model=backbone)
train_tf = timm.data.create_transform(**cfg, is_training=True, auto_augment="rand-m7-mstd0.5", re_prob=0.1)
eval_tf = timm.data.create_transform(**cfg)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.05)
# Training loop: same Manifest.csv, crop scale 2.7, CrossEntropy(label_smoothing=0.1), 10-15 epochs, AMP.